# Introduction to Geoecology with Python

Welcome to the Geoecology Sandbox! This notebook introduces key Python tools for analysing and visualising ecological and geospatial data.

## Contents
1. [Loading and exploring data](#1-loading-and-exploring-data)
2. [Descriptive statistics](#2-descriptive-statistics)
3. [Visualising distributions](#3-visualising-distributions)
4. [Relationships between environmental variables](#4-relationships-between-environmental-variables)
5. [Mapping species observations](#5-mapping-species-observations)
6. [Simple species richness analysis](#6-simple-species-richness-analysis)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point
import folium

# Set a consistent plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

## 1. Loading and exploring data

We load a small dataset of tree species observations across Central Europe. Each row records a species, its geographic coordinates, elevation, a vegetation index (NDVI), and basic climate variables.

In [ ]:
df = pd.read_csv("../data/species_observations.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.dtypes

## 2. Descriptive statistics

In [ ]:
df.describe()

In [ ]:
# Observation counts per species
df.groupby("species")["count"].sum().sort_values(ascending=False)

## 3. Visualising distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(
    axes,
    ["elevation_m", "temperature_c", "precipitation_mm"],
    ["Elevation (m)", "Mean temperature (°C)", "Annual precipitation (mm)"],
):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")

fig.suptitle("Distribution of environmental variables", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Relationships between environmental variables

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    df["elevation_m"],
    df["temperature_c"],
    c=df["precipitation_mm"],
    cmap="Blues",
    s=df["count"] * 5,
    edgecolors="grey",
    linewidths=0.5,
    alpha=0.85,
)
plt.colorbar(scatter, ax=ax, label="Precipitation (mm)")
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Mean temperature (°C)")
ax.set_title("Elevation vs. temperature (bubble size = observation count)")
plt.tight_layout()
plt.show()

In [ ]:
# Box plots of NDVI per species
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=df, x="species", y="ndvi", palette="Set2", ax=ax)
ax.set_xlabel("Species")
ax.set_ylabel("NDVI")
ax.set_title("NDVI distribution by species")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. Mapping species observations

We convert the CSV to a **GeoDataFrame** (point geometry) and create an interactive map with Folium.

In [ ]:
geometry = [Point(lon, lat) for lon, lat in zip(df["longitude"], df["latitude"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
gdf.head(3)

In [ ]:
# Colour palette per species
species_colours = {
    "Quercus robur": "green",
    "Fagus sylvatica": "blue",
    "Pinus sylvestris": "orange",
    "Betula pendula": "purple",
    "Picea abies": "darkgreen",
    "Alnus glutinosa": "red",
    "Quercus petraea": "cadetblue",
}

m = folium.Map(location=[50.5, 11.5], zoom_start=5, tiles="CartoDB positron")

for _, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=row["count"] / 2,
        color=species_colours.get(row["species"], "grey"),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['species']}</b><br>"
            f"Count: {row['count']}<br>"
            f"Elevation: {row['elevation_m']} m<br>"
            f"Temp: {row['temperature_c']} °C",
            max_width=200,
        ),
    ).add_to(m)

m

## 6. Simple species richness analysis

We bin the observations by 200 m elevation bands and compute the number of distinct species per band — a classic elevation–diversity relationship.

In [ ]:
df["elev_band"] = pd.cut(df["elevation_m"], bins=range(0, 1601, 200))

richness = (
    df.groupby("elev_band", observed=True)["species"]
    .nunique()
    .reset_index()
    .rename(columns={"species": "species_richness"})
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    range(len(richness)),
    richness["species_richness"],
    color="mediumseagreen",
    edgecolor="white",
)
ax.set_xticks(range(len(richness)))
ax.set_xticklabels(
    [str(b) for b in richness["elev_band"]], rotation=30, ha="right", fontsize=9
)
ax.set_xlabel("Elevation band (m)")
ax.set_ylabel("Species richness")
ax.set_title("Species richness along an elevation gradient")
plt.tight_layout()
plt.show()

---
**Next steps:** explore your own data, experiment with raster analysis using `rasterio` and `rasterstats`, or fit statistical models with `scipy` and `statsmodels`.